# DINOv3 Strong-Lens Finder — Results

This notebook runs the **whole pipeline end-to-end on CPU** using a `timm` fallback
backbone — **no DINOv3 download and no GPU required** — so you can confirm your
setup works and get a first ROC curve + ranked-candidate grid. Swap in the real
DINOv3 backbone + a GPU (last section) for publication-grade numbers.

**Run from the project root** after `pip install -e ".[sim,demo,notebook]"`.

In [ ]:
import os, sys
# fall back to the source tree if the package isn't pip-installed
if os.path.isdir('src'):
    sys.path.insert(0, 'src')
import numpy as np
import matplotlib.pyplot as plt
from dino_lens_finder.config import Config
from dino_lens_finder.report import roc_figure, ranked_grid, read_ranked

## 1. Simulated data
Builds a small physical (lenstronomy) set if available, else the dependency-free toy.

In [ ]:
DATA = 'data/demo'
if not os.path.exists(f'{DATA}/index.csv'):
    try:
        from dino_lens_finder.simulation.lenstronomy_sim import build_dataset
        build_dataset(DATA, n_train=400, n_val=120, pos_frac=0.2, size=64)
    except ImportError:
        from dino_lens_finder.simulation.toy import build_dataset
        build_dataset(DATA, n_train=400, n_val=120, pos_frac=0.2, size=64)
print('dataset ready ->', DATA)

## 2. A CPU-friendly config
Frozen `timm` ViT (ImageNet-pretrained) — proves the pipeline without DINOv3/GPU.

In [ ]:
cfg = Config.from_dict({
    'backbone': {'source': 'timm', 'name': 'vit_small_patch16_224',
                 'img_size': 64, 'prefix_tokens': 1, 'pretrained': True},
    'finetune': {'mode': 'frozen'},
    'head': {'hidden_dim': 256, 'dropout': 0.2},
    'data': {'index': f'{DATA}/index.csv', 'img_size': 64, 'num_workers': 0,
             'mean': [0.5, 0.5, 0.5], 'std': [0.5, 0.5, 0.5]},
    'train': {'epochs': 5, 'batch_size': 32, 'grad_accum': 1, 'amp': False,
              'balance_sampler': True, 'out_dir': 'runs/demo'},
    'eval': {'fpr_target': 0.1, 'top_n': 16},
})
cfg.backbone.name, cfg.finetune.mode

## 3. Train (CPU, a couple of minutes)

In [ ]:
from dino_lens_finder.training.trainer import Trainer
ckpt = Trainer(cfg).fit()
ckpt

## 4. Evaluate → metrics + ranked candidates

In [ ]:
from dino_lens_finder.evaluation import evaluate_checkpoint
metrics = evaluate_checkpoint(cfg, ckpt, split='val', out='runs/demo/ranked.csv')
metrics

## 5. ROC curve
Accuracy is meaningless under extreme imbalance — we read **AUC** and the
operating point **TPR@FPR**.

In [ ]:
rows = read_ranked('runs/demo/ranked.csv')
y = [r[2] for r in rows]; scores = [r[1] for r in rows]
fig = roc_figure(y, scores, cfg.eval.fpr_target)
plt.show()

## 6. Top-ranked candidates
Green border = true lens, red = false positive — a visual read of precision among
the candidates a human would inspect.

In [ ]:
fig = ranked_grid(rows, n=16, cols=4)
plt.show()

## 7. Getting publication-grade numbers

1. **DINOv3 backbone + GPU**: in `configs/default.yaml` keep `backbone.source: hf`
   (`facebook/dinov3-vitb16-pretrain-lvd1689m`) and `finetune.mode: lora`; run
   `dino-lens check-backbone` first.
2. **Full simulated set**: `dino-lens simulate-physical --n-train 4000 --n-val 1000 --pos-frac 0.1`.
3. **Train + evaluate**: `dino-lens train` then `dino-lens eval`.
4. **Benchmark**: `dino-lens make-bologna ...`, point `data.index` at
   `data/bologna/index.csv`, and re-run `dino-lens eval` — *train on simulation,
   evaluate on the real Bologna benchmark*.

Re-running this notebook with that config (and a GPU) reproduces the same plots
with real DINOv3 features.